In [36]:
import pandas as pd
import glob as glob
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import gpplot as gpp
import random
pd.options.mode.chained_assignment = None

Modeled off of code by Smriti Srikanth

Running the MAGeCK-MLE and MAGeCK-RRA hit calling methods on Brunello A375 data from Sanson et al 2018

Using MAGeCK v. 0.5.9.4

In [37]:
#importing data from data cleaned for comparison of Jacquere vs. Brunello screen performance 
screen_results=pd.read_csv("../../6. Jacquere Screen Performance/Data/brunelloA375lfc.csv",low_memory=False)
screen_results=screen_results.rename(columns={"sgRNA Sequence":"Construct Barcode"})
screen_results

,Unnamed: 0,Construct Barcode,sgRNA_lfc,Input,Quota,Target Taxon,Target Gene ID,Target Gene Symbol,Target Transcript,Target Alias,...,Other Target Matches,Aggregate CFD Score,Off-Target CFD100 Hits,Off-Target Tier I CFD100 Hits,On-Target Ruleset,On-Target Efficacy Score,On-Target Rank,Pick Order,Picking Round,Picking Notes
0,0,AAAAAAAATCCGGACAATGG,-1.450226,ENSG00000085491,1.0,9606.0,ENSG00000085491,SLC25A24,ENST00000565488.6,NaN,...,[],0.0,0,0,RS3seq-Chen2013+RS3target,-0.3000,114.0,4.0,0.0,Preselected
1,1,AAAAAAAGGATGGTGATCAA,-0.060651,ENSG00000124279,1.0,9606.0,ENSG00000124279,FASTKD3,ENST00000264669.10,NaN,...,[],1.1361,0,0,RS3seq-Chen2013+RS3target,0.1096,78.0,4.0,0.0,Preselected
2,2,AAAAAAATGACATTACTGCA,-2.610642,ENSG00000116752,1.0,9606.0,ENSG00000116752,BCAS2,ENST00000369541.4,NaN,...,[],0.8889,0,0,RS3seq-Chen2013+RS3target,0.1668,22.0,3.0,0.0,Preselected
3,3,AAAAAAATGTCAGTCGAGTG,0.546546,ENSG00000125245,1.0,9606.0,ENSG00000125245,GPR18,ENST00000397470.5,NaN,...,[],0.6154,0,0,RS3seq-Chen2013+RS3target,-0.3968,69.0,4.0,0.0,Preselected
4,4,AAAAAACACAAGCAAGACCG,0.449231,ENSG00000197016,1.0,9606.0,ENSG00000197016,ZNF470,ENST00000330619.13,NaN,...,[],0.8571,0,0,RS3seq-Chen2013+RS3target,-0.1512,123.0,3.0,0.0,Preselected
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86382,86382,TTTGTTGGATACGAAGTGAA,0.154829,ENSG00000120860,1.0,9606.0,ENSG00000120860,WASHC3,ENST00000240079.11,NaN,...,[],0.8667,0,0,RS3seq-Chen2013+RS3target,0.3445,8.0,2.0,0.0,Preselected
86383,86383,TTTGTTGGCACAAATACGGG,0.175347,ENSG00000165496,1.0,9606.0,ENSG00000165496,RPL10L,ENST00000298283.5,NaN,...,[],0.0,0,0,RS3seq-Chen2013+RS3target,-0.1607,70.0,4.0,0.0,Preselected
86384,86384,TTTGTTTCATCACATCGATG,0.166978,ENSG00000136011,1.0,9606.0,ENSG00000136011,STAB2,ENST00000388887.7,NaN,...,[],0.0,0,0,RS3seq-Chen2013+RS3target,0.4258,316.0,2.0,0.0,Preselected
86385,86385,TTTGTTTCCACAAACATGTA,0.086837,ENSG00000081870,1.0,9606.0,ENSG00000081870,IFT25,ENST00000194214.10,NaN,...,[],0.0694,0,0,RS3seq-Chen2013+RS3target,-0.6873,22.0,2.0,0.0,Preselected


Generate pseudogenes from negative (intergenic) controls to be scored with the MAGeCK hit-calling methods for empirical p-value calculation

In [38]:
num_pseudogene = 1000
genetargeting_results = (screen_results.loc[(screen_results['Target Gene Symbol']
                                            .str.contains('ONE_SITE_INTERGENIC')==False) &
                                              (screen_results['Target Gene Symbol']
                                            .str.contains('NO_SITE')==False) &
                                              (screen_results['Target Gene Symbol']
                                            .str.contains('MULTIPLE_INTERGENIC')==False),:]
                      .reset_index(drop=True))
# number of guides per pseudogene should match that of actual genes
n_guides_per_gene = genetargeting_results.groupby('Target Gene Symbol')['Construct Barcode'].nunique().mode()[0]
control_sgrnas = screen_results[screen_results["Target Gene Symbol"]=="ONE_INTERGENIC_SITE"]['Construct Barcode'].unique()
pseudogene_df_list = []

random.seed(0)
for i in range(num_pseudogene):
     chosen_guides = random.choices(control_sgrnas, k=n_guides_per_gene)
     temp_df_list = []
     for j in range(n_guides_per_gene):
         g = chosen_guides[j]
         temp_row = screen_results.loc[(screen_results['Construct Barcode'] == g) &
                                           (screen_results['Target Gene Symbol'].str.contains('ONE_INTERGENIC_SITE')),:]
         temp_row.loc[:,'Construct Barcode with identifier'] = g + 'p' + str(i) + 'g' + str(j)
         temp_df_list.append(temp_row)
     temp_df = pd.concat(temp_df_list)
     temp_df.loc[:,'Target Gene Symbol'] = 'Pseudogene_' + str(i)
     pseudogene_df_list.append(temp_df)

pseudogene_df = pd.concat(pseudogene_df_list).reset_index(drop=True)
screen_results_withpseudogenes = pd.concat([screen_results,pseudogene_df]).reset_index(drop=True)

screen_results_withpseudogenes["Construct Barcode with identifier"]=screen_results_withpseudogenes.apply(lambda x: x["Construct Barcode"] if type(x["Construct Barcode with identifier"])==float else x["Construct Barcode with identifier"],axis=1)
#remove NA target gene symbols
screen_results_withpseudogenes["Target Gene Symbol"]=screen_results_withpseudogenes.apply(lambda x:x["Target Gene ID"] if type(x["Target Gene Symbol"])==float else x["Target Gene Symbol"],axis=1)

### Generating input files
Must feature raw read counts for each construct along with its corresponding target.

In [39]:
mageck_guide_gene_annotation = screen_results_withpseudogenes[['Construct Barcode','Construct Barcode with identifier','Target Gene Symbol']].drop_duplicates()

#read in raw results
reads_df= pd.read_excel("../../6. Jacquere Screen Performance/Data/Brunello_Sanson2018_SuppData1.xlsx",sheet_name="A375_mod_tracr raw reads",skiprows=1)
reads_df=reads_df.rename(columns={"sgRNA Sequence":"Construct Barcode"})

#must merge on the guide sequence without pseudogene identifer appended because that is what is present in reads_df
annotated_reads_df = mageck_guide_gene_annotation.merge(reads_df,how = 'left',on = 'Construct Barcode')

#MAGeCK throws an error when the same guide targets multiple genes, so make construct barcode unique by appending gene symbol
annotated_reads_df['Construct_Barcode_with_identifier'] = annotated_reads_df['Construct Barcode with identifier'] + annotated_reads_df['Target Gene Symbol']
#also throws an error if column names have spaces
annotated_reads_df=annotated_reads_df.rename(columns={"Target Gene Symbol":"target_gene"})

annotated_reads_df[['Construct_Barcode_with_identifier','target_gene', "pDNA",'RepA','RepB',"RepC"]].to_csv('../Data/hitcalling_methods_inputs/mageck_input_brunello_A375counts.txt', sep='\t', index=False)

control_sgrnas_raw = annotated_reads_df[annotated_reads_df["Construct Barcode"].isin(control_sgrnas)]
control_sgrnas_raw = control_sgrnas_raw[["Construct Barcode"]].drop_duplicates()
control_sgrnas_raw["Construct Barcode"]=control_sgrnas_raw["Construct Barcode"]+"ONE_INTERGENIC_SITE"
control_sgrnas_raw.to_csv('../Data/hitcalling_methods_inputs/mageck_input_brunello_controls_A375.txt', index=False, header = False)


### Running MAGeCK-RRA

mageck test -k mageck_input_brunello_A375counts.txt -t RepA,RepB,RepC -c pDNA -n brunello --control-sgrna mageck_input_brunello_controls_A375.txt --norm-method total -n brunello_A375_mageck_RRA_results 

### Running MAGeCK-MLE 
Skipping the permutation step, which determines empirical p-values with the set of all targeting sgRNAs as the null distribution, since I am computing this manually just using the intergenic controls as the null distribution. 

mageck mle -k mageck_input_brunello_A375counts.txt --day0-label pDNA --control-sgrna mageck_input_brunello_controls_A375.txt --norm-method total -n brunello_A375_mageck_MLE_results --no-permutation-by-group